# Verification — `MIL_CREDA` against `research-concept-r17.md`

Executed report, not the source of truth. Every mathematical claim lives as a
test under `tests/`; this page runs them and shows the evidence. Existence of
this file proves nothing — `verify` reads whether its cells actually ran.

The implementation computes with PyTorch. That is a change of backend, not of
method: the equations are the same equations, and this report re-establishes them
over the same 200 sweep configurations as the numpy pass did, so a difference
would be attributable to the backend and to nothing else.

Re-run headlessly from the repository root:

```
.venv/bin/jupyter nbconvert --to notebook --execute --inplace \
  MIL-CREDA/Notebooks/verification.ipynb
```

In [ ]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [ ]:
import ast
import sys
from pathlib import Path

import torch

# `ROOT` comes from the cell above -- the forge's own, travelling byte for
# byte -- and is not worked out again here.
sys.path.insert(0, str(ROOT / "src"))
print(f"torch {torch.__version__}   default dtype for this package: float64")

# Which revision each module was written against, read statically.
for file in sorted((ROOT / "src" / "MIL_CREDA").rglob("*.py")):
    if file.name == "__init__.py":
        continue
    for node in ast.parse(file.read_text()).body:
        # Both assignment forms: `__provenance__ = {...}` (ast.Assign, whose
        # names live in `targets`) and `__provenance__: dict = {...}`
        # (ast.AnnAssign, one `target`). Reading only `targets` skips the
        # annotated form silently, and a module then reports no provenance
        # while declaring one perfectly.
        names = getattr(node, "targets", None) or [getattr(node, "target", None)]
        if any(getattr(t, "id", None) == "__provenance__" for t in names if t):
            p = ast.literal_eval(node.value)
            print(f"{file.name:<18} {p['revision']:<26} eqs {','.join(p['equations'])}")

## Levels 1, 2, 4 and 5 — the suite

A red cell here means the implementation does not match what the proposal
claims. Do not edit the assertion to make it pass.

In [ ]:
import pytest

code = pytest.main(["-q", str(ROOT / "tests"), "--rootdir", str(ROOT)])
assert code == 0, f"test suite failed (pytest exit code {code})"

## Level 3 — synthetic evidence

Deterministic, fixed seed, ground truth known by construction. State the
expected behaviour before looking at the output.

In [ ]:
# Level 3 evidence: the bound r16 adopted, drawn from the same sweep the tests use.
import sys
sys.path.insert(0, str(ROOT / "tests"))
import matplotlib.pyplot as plt
from MIL_CREDA_Benchmark import figures
from sweep import sweep

distances = torch.cat([c["squared_distances"] for c in sweep() if c["squared_distances"].numel()])
largest = float(distances.max())
print(f"target bags measured: {distances.numel()}   max d_j^2 = {largest:.6f}")
print(f"adopted normalizer /2 -> max l_loc,j = {largest / 2:.6f}   (uses the full range)")
print(f"retired normalizer /4 -> max l_loc,j = {largest / 4:.6f}   (upper half unusable)")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(distances.numpy(), bins=60, color="#4c72b0")
ax.axvline(2.0, color="#c44e52", lw=2, label="adopted bound $d_j^2 < 2$  (r16, Eq. 38)")
ax.axvline(4.0, color="#8172b2", lw=2, ls="--", label="retired bound $d_j^2 \\leq 4$  (r14)")
ax.set_xlim(0, 4.2)
ax.set_xlabel("$d_j^2$  (Eq. 31)")
ax.set_ylabel("target bags")
ax.set_title("Local discrepancy over 200 sweep configurations (PyTorch)")
ax.legend()
fig.tight_layout()
display(figures.inline(figures.emit(
    fig, ROOT / "MIL-CREDA" / "Results" / "local_distance_bound")))

In [ ]:
# El sello: contra qué código corrió este informe. Sin él, un informe viejo y uno
# recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())